In [50]:
from dotenv import load_dotenv
load_dotenv()

True

In [51]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # loads ANTHROPIC_API_KEY from .env automatically

client = Anthropic()

In [52]:
from anthropic import Anthropic
from anthropic.types import MessageParam

def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            MessageParam(role="user", content=prompt)
        ]
    )
    return response.content[0].text

In [53]:
def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-5",  # correct model name
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}] # type: ignore
    )
    return response.content[0].text

In [54]:
llm("Hello World")

'Hello! 👋 Welcome! How can I help you today?'

In [55]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

answer = llm(prompt)
print(answer)

I'd be happy to help you! However, I need a bit more information to give you the best answer:

1. **Which course are you interested in?** (There may be multiple courses available)
2. **When does the course start?** (or has it already started?)
3. **Is there a registration deadline?**

In general, here are some things to consider:

- **Many courses allow late registration** during the first week or two
- **Some online courses** have rolling admissions and you can join anytime
- **Academic courses** at universities often have strict add/drop deadlines
- You may need to check with the **instructor or administrator** about catching up on missed content

**What you can do right now:**
- Contact the course instructor or administrative office directly
- Check the course website or syllabus for registration policies
- Ask if there's any prerequisite material you'd need to review

If you let me know which specific course you're asking about, I can give you more targeted advice!
Based on the con

In [56]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
# make a get request to url
response = requests.get(docs_url)

# parse the response body as JSON. give sus a python list/dict
courses_raw = response.json()

print(courses_raw)


documents = []
url_prefix = 'https://datatalks.club/faq' # base url.

for course in courses_raw:
    # build the full URL for this course's FAQ
    course_url = f'{url_prefix}{course['path']}'

# fetch the FAQ data for this specific course.
    course_response = requests.get(course_url)

    # if request failed (400, 500 etc) stop and raise an error immediately.
    course_response.raise_for_status()

    #parse this course FAQ response as JSON.
    course_data = course_response.json()

# Add all FAQ documents from this course into our master list.
    # extend() adds each item individually
    documents.extend(course_data)

# shows the number of FAQ documents which we collect across all courses.
len(documents)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}]


1208

In [57]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [58]:
documents[1100]

{'id': '841966c903',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'Where is the FAQ for Prefect questions?',
 'answer': '[Here](https://docs.google.com/document/d/1Nyktf7WoRec5lDUBREXL5zLI1Edbw9_R8e45fDn4KB8/edit?usp=sharing)'}

In [59]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [60]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [61]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [62]:
search_results = search(question)

In [63]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [64]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [65]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [66]:
# go over all the results from search =, we want to turn the dictionary into context search results.
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [67]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [68]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [75]:
def llm(instructions, user_prompt, model ="claude-sonnet-4-5" ):
    message_history = [
        {"role": "user", "content": user_prompt}
    ]

    response = client.messages.create(
        model= model,
        max_tokens=1024,
        system=instructions,
        messages=message_history  # type: ignore
    )
    return response.content[0].text


def get_cost(response):
    input_price  = 3.00 / 1_000_000   # claude-sonnet-4-5 input price per token
    output_price = 15.00 / 1_000_000  # claude-sonnet-4-5 output price per token

    cost = (
        response.usage.input_tokens * input_price +
        response.usage.output_tokens * output_price
    )
    return cost


In [79]:
def rag(query, model="claude-sonnet-4-5"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


answer = rag("how many curses are present?")
print(answer)

Based on the provided context, I don't see any mention of "curses" in the course materials. The context contains information about course modules, embeddings, database operations, chunking strategies, and frameworks like Langchain, but there is no reference to curses.

**I don't know** how many curses are present, as this information is not available in the given context.

If you meant to ask about "courses" instead of "curses," I can see references to course modules (Module 1, Module 3) and workshops, but the complete number of courses is not explicitly stated in this context.
